## Section 1. Introduction to the Problem/Task

### Problem Statement
Medical professionals and researchers frequently need to query large, complex clinical guideline documents such as the **2024 European Society of Cardiology (ESC) Guidelines**. These documents span hundreds of pages and contain dense, technical information covering diagnostic criteria, treatment algorithms, risk stratification scores, and drug recommendations.

Manually searching through such documents is time-consuming and error-prone. This project addresses that challenge by building a **Retrieval-Augmented Generation (RAG) system** that allows users to ask natural language questions and receive accurate, context-grounded answers — complete with source page citations.

### Approach
This notebook implements a full RAG pipeline using **Alibaba Qwen3-4B-Instruct** (a 4B parameter model with strong multilingual and instruction-following capabilities) as the language model backbone:

1. **PDF Ingestion** — The 2024 ESC Guidelines PDF is parsed and chunked into overlapping text segments.
2. **Embedding** — Chunks are embedded using **MedCPT** (a biomedical bi-encoder) and stored in a **FAISS** vector index.
3. **Retrieval & Reranking** — At query time, the top candidate chunks are retrieved and reranked using a **Cross-Encoder** for precision.
4. **Generation** — The reranked context is fed to the LLM to produce a final answer.
5. **Evaluation** — Responses are scored using ROUGE, BERTScore, Semantic Similarity, and LLM-as-a-judge metrics.
6. **Deployment** — The system is served through an interactive **Gradio** web interface.

## Section 2. Dataset Description

### Source Document
The dataset used in this project is the **2024 ESC Guidelines on Atrial Fibrillation** (and related cardiovascular conditions), published by the European Society of Cardiology. It is a comprehensive clinical reference document containing:

- **Treatment recommendations** classified by evidence level (Class I, IIa, IIb, III)
- **Diagnostic criteria** for cardiovascular conditions
- **Risk scores** (CHA₂DS₂-VASc, HAS-BLED, etc.)
- **Drug dosing tables** and contraindication summaries
- **Structured algorithms** for clinical decision-making

The PDF is dense, multi-columnar, and includes both prose and tabular data — making it a challenging but representative real-world retrieval task.

### 2.1 Dataset Cleaning

The raw PDF is loaded using **PyMuPDF** (`langchain_community.document_loaders.PyMuPDFLoader`), which efficiently extracts text from each page while preserving layout structure. The documents are then split into overlapping chunks using `RecursiveCharacterTextSplitter` with a chunk size of 512 and overlap of 64 tokens.

In [1]:
# Install core libraries
!pip install torch transformers accelerate bitsandbytes langchain-community langchain-text-splitters langchain-core faiss-cpu pymupdf sentence-transformers

# Install UI and metrics libraries
!pip install -q gradio
!pip install -q rouge-score bert-score nltk pandas

print("-" * 50)
print("✅ ALL LIBRARIES INSTALLED")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Preparing metad

In [2]:
# ==========================================
# IMPORTS & CONFIGURATION
# ==========================================
import torch, gc, os, csv, time, re, string, collections
import pandas as pd
import warnings
from datetime import datetime
from typing import List
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoModel
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

warnings.filterwarnings("ignore")

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
HF_TOKEN = os.getenv("HF_TOKEN")
PDF_PATH = "/content/2024ESC-compressed.pdf"

In [3]:
# ==========================================
# PDF LOADER (PyMuPDF)
# ==========================================
print("\n📂 Loading and indexing PDF with MedCPT...")
loader = PyMuPDFLoader(PDF_PATH)
documents = loader.load()
print(f"✅ Loaded {len(documents)} pages from PDF.")


📂 Loading and indexing PDF with MedCPT...
✅ Loaded 101 pages from PDF.


### 2.2 Comparing Embedding Models

For document and query embedding, we use **MedCPT** (Medical Contrastive Pre-Training), a biomedical bi-encoder from NCBI trained on PubMed data. It uses separate encoders for queries and documents, making it well-suited for asymmetric retrieval tasks like this one.

MedCPT was chosen over general-purpose models (e.g., `all-MiniLM-L6-v2`) because of its domain-specific pretraining on medical literature, which produces embeddings that better capture clinical terminology and concept relationships.

In [4]:
# ==========================================
# MEDCPT EMBEDDINGS (BI-ENCODER)
# ==========================================
class MedCPTEmbeddings(Embeddings):
    def __init__(self, load_article_encoder: bool = True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("⚙️ Initializing MedCPT Query Encoder...")
        self.models = {
            "qry_tok": AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder"),
            "qry_mod": AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder"),
        }
        if load_article_encoder:
            print("⚙️ Initializing MedCPT Article Encoder...")
            self.models["art_tok"] = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
            self.models["art_mod"] = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder").to(self.device)

    def embed_documents(self, texts):
        all_embeddings = []
        for i in range(0, len(texts), 8):
            batch = texts[i : i + 8]
            inputs = self.models["art_tok"](batch, max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
            with torch.no_grad():
                output = self.models["art_mod"](**inputs)
                all_embeddings.extend(output.last_hidden_state[:, 0, :].cpu().tolist())
        return all_embeddings

    def embed_query(self, text):
        self.models["qry_mod"].to(self.device)
        inputs = self.models["qry_tok"]([text], max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
        with torch.no_grad():
            output = self.models["qry_mod"](**inputs)
            result = output.last_hidden_state[:, 0, :][0].cpu().tolist()
        self.models["qry_mod"].to("cpu")
        del inputs, output
        torch.cuda.empty_cache()
        return result

    def unload_article_encoder(self):
        if "art_mod" in self.models:
            print("🧹 Optimizing Memory: Removing Article Encoder...")
            del self.models["art_mod"], self.models["art_tok"]
            torch.cuda.empty_cache()
            gc.collect()

# ==========================================
# CHUNK & INDEX
# ==========================================
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
chunks = splitter.split_documents(documents)

medcpt_embeddings = MedCPTEmbeddings(load_article_encoder=True)
vector_store = FAISS.from_documents(chunks, medcpt_embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 8})
print(f"✅ Indexed {len(chunks)} chunks.")
medcpt_embeddings.unload_article_encoder()

⚙️ Initializing MedCPT Query Encoder...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Indexed 1642 chunks.
🧹 Optimizing Memory: Removing Article Encoder...


## Section 3. Requirements

The following libraries are required to run this notebook. They cover deep learning (PyTorch, Transformers), RAG pipeline components (LangChain, FAISS), PDF parsing, and evaluation metrics (ROUGE, BERTScore, Gradio UI).

In [5]:
# Install core libraries
#!pip install torch transformers accelerate bitsandbytes langchain-community langchain-text-splitters langchain-core faiss-cpu pymupdf sentence-transformers

# Install UI and metrics libraries
#!pip install -q gradio
#!pip install -q rouge-score bert-score nltk pandas

## Section 4. System Architecture

The RAG pipeline is composed of four core components:

1. **Vector Store (FAISS)** — Stores MedCPT embeddings of document chunks for fast approximate nearest-neighbor retrieval.
2. **Cross-Encoder Reranker** — Re-scores the top-k retrieved chunks using `BAAI/bge-reranker-base`.
3. **Language Model (LLM)** — Qwen3-4B-Instruct generates the final answer conditioned on reranked context.
4. **Semantic Similarity Model** — `all-MiniLM-L6-v2` is loaded on CPU for offline metric computation.

All models are loaded with **4-bit quantization (NF4)** via BitsAndBytes.

In [6]:
print("\n⚖️ Loading Cross-Encoder (Reranker)...")
reranker = CrossEncoder("BAAI/bge-reranker-base", device="cpu")

print("⚙️ Loading Semantic Similarity Model...")
metric_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

print("\n🚀 Loading Qwen-3 LLM...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
)
print("✅ Qwen LLM ready.\n")


⚖️ Loading Cross-Encoder (Reranker)...


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✅ Qwen LLM ready.



## Section 5. System Evaluation (Unseen Queries)

The system is evaluated using a multi-dimensional set of metrics to capture different aspects of response quality:

| Metric Group | Metrics | Description |
|---|---|---|
| **Lexical** | ROUGE-1, ROUGE-2, ROUGE-L | N-gram overlap with reference answer |
| **Semantic** | BERTScore (P/R/F1), Cosine Similarity | Embedding-space similarity |
| **RAG-specific** | Faithfulness, Answer Relevancy, Context Recall | LLM-as-a-judge (0–10 scale) |

These metrics collectively assess whether responses are factually grounded, relevant to the query, and linguistically similar to expected answers.

In [7]:
def normalize_text(s):
    if pd.isna(s): return ""
    s = str(s).lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def calc_accuracy_metrics(pred, truth):
    norm_pred = normalize_text(pred).split()
    norm_truth = normalize_text(truth).split()
    exact_match = int(norm_pred == norm_truth)
    common = collections.Counter(norm_pred) & collections.Counter(norm_truth)
    num_same = sum(common.values())
    if len(norm_pred) == 0 or len(norm_truth) == 0:
        f1 = precision = recall = int(norm_pred == norm_truth)
    elif num_same == 0:
        f1 = precision = recall = 0.0
    else:
        precision = num_same / len(norm_pred)
        recall = num_same / len(norm_truth)
        f1 = (2 * precision * recall) / (precision + recall)
    return exact_match, round(precision, 4), round(recall, 4), round(f1, 4)

def calc_lexical_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0, 0, 0, 0, 0, 0
    smoothie = SmoothingFunction().method4
    ref = [normalize_text(truth).split()]
    hyp = normalize_text(pred).split()
    b1 = sentence_bleu(ref, hyp, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    b2 = sentence_bleu(ref, hyp, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    b4 = sentence_bleu(ref, hyp, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(str(truth), str(pred))
    return round(b1,4), round(b2,4), round(b4,4), round(rouge_scores['rouge1'].fmeasure,4), round(rouge_scores['rouge2'].fmeasure,4), round(rouge_scores['rougeL'].fmeasure,4)

def calc_semantic_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0,0,0,0
    P, R, F1 = bert_score([str(pred)], [str(truth)], lang="en", model_type="distilbert-base-uncased", device="cpu")
    embeddings = metric_model.encode([str(pred), str(truth)], convert_to_tensor=True)
    cosine_sim = float(util.pytorch_cos_sim(embeddings[0], embeddings[1])[0][0])
    return round(P.item(),4), round(R.item(),4), round(F1.item(),4), round(cosine_sim,4)

def calc_rag_metrics_llm(query, context, response, expected):
    messages = [
        {"role": "system", "content": (
            "You are an expert RAG evaluator. Output ONLY three comma-separated numbers between 0 and 10 (e.g., '9, 8, 10').\n"
            "1. Faithfulness: Is the Response completely grounded in the Context?\n"
            "2. Answer Relevancy: Does the Response address the Query?\n"
            "3. Context Recall: Is the Expected Answer found within the Context?"
        )},
        {"role": "user", "content": f"Query: {query}\nExpected Answer: {expected}\nContext: {context}\nResponse: {response}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=15, temperature=0.01)
    input_length = inputs["input_ids"].shape[1]
    score_text = tokenizer.decode(output[0][input_length:], skip_special_tokens=True).strip()
    del inputs, output
    torch.cuda.empty_cache()
    try:
        scores = [int(re.search(r'\d+', s).group()) for s in score_text.split(',')]
        if len(scores) >= 3:
            return scores[0], scores[1], scores[2]
    except:
        pass
    return 0, 0, 0

In [8]:
def rag_query_stream(query: str, expected: str = ""):
    yield "⏳ **Status:** 🔍 Retrieving relevant documents from VectorDB...\n\n---\n"
    candidates = retriever.invoke(query)

    yield "⏳ **Status:** 📊 Reranking top documents with CrossEncoder...\n\n---\n"
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in ranked[:4]]
    context = "\n\n".join(doc.page_content for doc in top_docs)
    pages = ", ".join(str(doc.metadata.get("page", "?")) for doc in top_docs)

    yield "⏳ **Status:** 🧠 Synthesizing answer with Qwen (this may take a moment)...\n\n---\n"
    messages = [
        {"role": "system", "content": (
            "You are a medical expert assistant specialising in cardiology. "
            "Answer the user's question using ONLY the context provided below. "
            "If the answer is not in the context, say you don't know.\n\n"
            f"Context:\n{context}"
        )},
        {"role": "user", "content": query},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen_start = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.05,
        )
    gen_time = time.time() - gen_start
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = len(generated_ids[0]) - input_length
    tokens_per_sec = generated_tokens / gen_time if gen_time > 0 else 0
    answer = tokenizer.decode(generated_ids[0][input_length:], skip_special_tokens=True)
    del inputs, generated_ids
    torch.cuda.empty_cache()

    yield "⏳ **Status:** 📈 Calculating evaluation metrics...\n\n---\n"
    exact_match, tok_p, tok_r, tok_f1 = calc_accuracy_metrics(answer, expected)
    bleu1, bleu2, bleu4, rouge1, rouge2, rougeL = calc_lexical_metrics(answer, expected)
    bert_p, bert_r, bert_f1, cos_sim = calc_semantic_metrics(answer, expected)
    faithfulness, answer_rel, ctx_recall = calc_rag_metrics_llm(query, context, answer, expected)

    metadata = f"**Source Pages:** {pages}\n\n"
    metrics_text = (f"**Latency:** {round(gen_time, 2)}s | "
                    f"**Speed:** {round(tokens_per_sec, 2)} Tokens/sec | "
                    f"**RAG Scores (F/R/C):** {faithfulness}/10, {answer_rel}/10, {ctx_recall}/10")
    yield f"{answer}\n\n---\n{metadata}{metrics_text}"

## Section 6. User Interface (Gradio)

The system is wrapped in an interactive web UI built with **Gradio**. Users can type clinical questions and receive streamed responses with real-time status updates for each pipeline stage (retrieval → reranking → generation → evaluation).

The interface uses a **Violet/Indigo** Qwen theme.

In [9]:
import gradio as gr

def gradio_wrapper(query):
    if not query or not query.strip():
        yield "⚠️ Please enter a valid question."
        return
    for status_update in rag_query_stream(query, expected=""):
        yield status_update

qwen_theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="indigo",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont('Inter'), 'ui-sans-serif', 'system-ui', 'sans-serif']
).set(
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_700",
    block_title_text_color="*primary_700",
    block_label_text_color="*primary_600"
)

with gr.Blocks(theme=qwen_theme) as demo:
    gr.Markdown("# 🌌 Cardiology AI Assistant (ESC 2024)")
    gr.Markdown("### ⚡ Powered by Alibaba Qwen-3")
    gr.Markdown("Ask questions based on the 2024 ESC Medical Guidelines. The system uses RAG with MedCPT embeddings, Cross-Encoder reranking, and Qwen generation.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Your Clinical Question",
                placeholder="e.g., What are the class I recommendations for anticoagulation in AF?",
                lines=3
            )
            submit_btn = gr.Button("Analyze Guidelines", variant="primary")

    output_text = gr.Markdown(label="Assistant Response")

    gr.Examples(
        examples=[
            "What are the class I recommendations for anticoagulation in AF?",
            "Summarize the treatment algorithm for chronic heart failure.",
            "What is the target LDL-C for very high-risk patients?"
        ],
        inputs=input_text
    )
    submit_btn.click(gradio_wrapper, inputs=input_text, outputs=output_text)

### Section 6.1 Deployed Web Application

The Gradio app is launched with `share=True` to generate a public Hugging Face Spaces-style URL. Queue mode is enabled to support streaming (yielded) responses.

In [ ]:
demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bef5bb25fca5aac409.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Section 6.2 Evaluation Results — Per-Query Data

The table below presents the full per-query evaluation results for **Qwen3-4B-Instruct** across all 20 test queries.

| Query | Pages | ROUGE-1 | ROUGE-2 | ROUGE-L | BERTScore P | BERTScore R | BERTScore F1 | Semantic Similarity | Faithfulness | Answer Relevancy | Context Recall | Latency (s) | Tokens/sec |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| What are the four essential treatment pillars of the AF-CARE framework? | 18, 7, 18, 7 | 0.5432 | 0.4051 | 0.4938 | 0.8102 | 0.8467 | 0.8281 | 0.5799 | 10 | 10 | 10 | 11.27 | 7.28 |
| What is the minimum ECG duration required to establish a diagnosis of clinical AF? | 13, 13, 60, 65 | 0.1509 | 0.0784 | 0.1132 | 0.7242 | 0.7197 | 0.7219 | 0.2791 | 10 | 10 | 9 | 5.38 | 8.18 |
| When is oral anticoagulation (OAC) recommended based on the CHA2DS2-VA score? | 67, 28, 46, 46 | 0.2667 | 0.1186 | 0.2333 | 0.6699 | 0.8169 | 0.7361 | 0.4476 | 10 | 10 | 10 | 18.95 | 8.39 |
| Which drugs are recommended as first-choice for rate control in patients with LVEF>40%? | 9, 66, 37, 66 | 0.3636 | 0.3226 | 0.3636 | 0.8029 | 0.9294 | 0.8615 | 0.6967 | 10 | 10 | 10 | 5.45 | 8.08 |
| Is antiplatelet therapy recommended as an alternative to OAC for stroke prevention in AF? | 8, 66, 9, 32 | 0.198 | 0.1414 | 0.198 | 0.7728 | 0.8089 | 0.7904 | 0.7786 | 10 | 10 | 10 | 15.08 | 8.29 |
| What is the recommended target for alcohol consumption to reduce AF recurrence? | 65, 11, 25, 11 | 0.7273 | 0.5806 | 0.6667 | 0.8734 | 0.9183 | 0.8953 | 0.7354 | 10 | 10 | 10 | 3.76 | 7.18 |
| For patients on VKA, what is the recommended target INR range and TTR percentage? | 30, 52, 35, 31 | 0.6364 | 0.4762 | 0.6364 | 0.8893 | 0.9177 | 0.9033 | 0.7513 | 10 | 10 | 10 | 5.63 | 7.11 |
| When should a "wait-and-see" approach be considered for cardioversion? | 9, 41, 39, 40 | 0.1449 | 0.0441 | 0.087 | 0.7133 | 0.8222 | 0.7638 | 0.6107 | 10 | 10 | 10 | 20.17 | 8.13 |
| List the three criteria used to decide on a dose reduction for Apixaban. | 31, 63, 51, 8 | 0.6061 | 0.4375 | 0.5758 | 0.8413 | 0.9065 | 0.8727 | 0.588 | 10 | 10 | 10 | 8.3 | 7.83 |
| What is the primary indication for long-term rhythm control therapy in AF patients? | 63, 67, 3, 13 | 0.5641 | 0.4324 | 0.5641 | 0.8295 | 0.8906 | 0.859 | 0.4969 | 10 | 10 | 10 | 4.03 | 7.94 |
| Is routine heart rhythm assessment recommended for individuals aged 65 and older? | 68, 60, 25, 60 | 0.4324 | 0.4167 | 0.4324 | 0.8318 | 0.9204 | 0.8739 | 0.8392 | 10 | 10 | 10 | 8.85 | 7.8 |
| Which heart failure medication is recommended for AF patients regardless of LVEF? | 28, 37, 67, 25 | 0.0 | 0.0 | 0.0 | 0.655 | 0.7206 | 0.6862 | 0.1979 | 1 | 10 | 0 | 12.87 | 8.16 |
| What should be done if a left atrial thrombus is detected prior to a planned cardioversion? | 41, 49, 41, 66 | 0.2921 | 0.092 | 0.1798 | 0.7423 | 0.8701 | 0.8011 | 0.6227 | 9 | 8 | 7 | 14.14 | 8.2 |
| Name two non-cardiac conditions associated with trigger-induced AF according to Table 14. | 3, 14, 15, 19 | 0.2857 | 0.1667 | 0.2449 | 0.7916 | 0.811 | 0.8012 | 0.8815 | 0 | 0 | 0 | 11.14 | 8.08 |
| What is the recommended weight loss target for obese individuals with AF? | 11, 65, 25, 62 | 0.5625 | 0.4 | 0.5 | 0.8083 | 0.8792 | 0.8423 | 0.4948 | 10 | 10 | 10 | 3.83 | 7.83 |
| Can a physician use a single-lead ECG to provide a definite diagnosis of AF? | 60, 10, 14, 14 | 0.2198 | 0.0778 | 0.1648 | 0.7487 | 0.8681 | 0.804 | 0.7491 | 10 | 10 | 10 | 25.59 | 8.32 |
| What are the symptoms of AF listed in Figure 1? | 15, 19, 49, 3 | 0.0488 | 0.0 | 0.0488 | 0.6799 | 0.7501 | 0.7133 | 0.3616 | 0 | 0 | 0 | 9.96 | 8.13 |
| Is it recommended to use a bleeding risk score to decide whether to withhold OAC? | 35, 66, 34, 53 | 0.3256 | 0.2381 | 0.3256 | 0.7829 | 0.9085 | 0.841 | 0.8273 | 10 | 10 | 10 | 11.39 | 8.07 |
| What is the definition of "Permanent AF" according to the guidelines? | 13, 13, 60, 13 | 0.8077 | 0.76 | 0.8077 | 0.8944 | 0.9527 | 0.9227 | 0.8034 | 10 | 10 | 10 | 4.68 | 7.48 |
| For patients undergoing cardiac surgery, when is surgical closure of the LAA recommended? | 11, 66, 47, 33 | 0.6667 | 0.5652 | 0.6667 | 0.8079 | 0.9098 | 0.8558 | 0.5857 | 10 | 10 | 10 | 6.85 | 7.59 |

## Section 7. Results and Analysis

This section presents the evaluation results for the **Qwen3-4B-Instruct** RAG system, tested on **20 unseen clinical queries** drawn from the 2024 ESC Guidelines on Atrial Fibrillation.

---

### 7.1 Evaluation Metrics Summary

**Lexical Metrics**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| ROUGE-1 | 0.3921 | 0.2350 | 0.0000 | 0.8077 |
| ROUGE-2 | 0.2877 | 0.2209 | 0.0000 | 0.7600 |
| ROUGE-L | 0.3651 | 0.2389 | 0.0000 | 0.8077 |

**Semantic Metrics**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| BERTScore Precision | 0.7835 | 0.0701 | 0.6550 | 0.8944 |
| BERTScore Recall | 0.8584 | 0.0695 | 0.7197 | 0.9527 |
| BERTScore F1 | 0.8187 | 0.0670 | 0.6862 | 0.9227 |
| Semantic Similarity | 0.6164 | 0.1915 | 0.1979 | 0.8815 |

**RAG-Specific Metrics (LLM-as-Judge, /10)**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| Faithfulness | 8.50 | 3.53 | 0.0 | 10.0 |
| Answer Relevancy | 8.90 | 3.08 | 0.0 | 10.0 |
| Context Recall | 8.30 | 3.64 | 0.0 | 10.0 |

**Performance**

| Metric | Mean | Std | Min | Max |
|--------|------|-----|-----|-----|
| Latency (s) | 10.366 | 6.053 | 3.760 | 25.590 |
| Tokens/sec | 7.904 | 0.386 | 7.110 | 8.390 |

---

### 7.2 Analysis

Qwen3-4B-Instruct achieved **the fastest generation speed** of all three models at **7.90 tokens/sec** — nearly double that of Llama-3 — while maintaining competitive quality across all metric categories. Its Answer Relevancy of **8.90/10** matches Llama-3's best score, and its Faithfulness of **8.50/10** and Context Recall of **8.30/10** confirm reliable grounding in retrieved ESC guideline content.

On lexical metrics, Qwen3 scored a ROUGE-L of **0.3651** and BERTScore F1 of **0.8187**, both slightly below Llama-3 but ahead of Phi-3 on ROUGE. The Semantic Similarity mean of **0.6164** — the lowest of the three — reflects Qwen3's tendency to rephrase guideline content rather than echo its exact phrasing, which is not necessarily a weakness in a clinical QA context.

The average latency of **10.37 seconds** is the lowest overall, though the high standard deviation (±6.05s) reveals considerable variability — some queries resolved in under 4 seconds while others extended to ~26 seconds, likely driven by differences in generated response length across query types.

---

### 7.3 Limitations
- Qwen3's high latency variance (std ±6.05s) may affect predictability in real-time deployment scenarios.
- Context Recall std of 3.64 suggests occasional retrieval misses on edge-case queries.
- Evaluation was conducted on 20 queries; a larger test set would improve statistical confidence.

---

## Section 7 (Continued). Cross-Model Comparative Analysis

The following analysis compares all three RAG systems — **Llama-3-8B-Instruct**, **Phi-3-Mini-4k-Instruct**, and **Qwen3-4B-Instruct** — evaluated on the same 20 unseen clinical queries from the 2024 ESC Guidelines on Atrial Fibrillation.

---

### Cross-Model Metrics Overview

| Metric | Llama-3-8B | Phi-3-Mini | Qwen3-4B |
|--------|-----------|------------|----------|
| ROUGE-1 | **0.4564** | 0.3860 | 0.3921 |
| ROUGE-2 | **0.3366** | 0.2663 | 0.2877 |
| ROUGE-L | **0.4072** | 0.3304 | 0.3651 |
| BERTScore F1 | **0.8351** | 0.8244 | 0.8187 |
| Semantic Similarity | **0.6647** | 0.6282 | 0.6164 |
| Faithfulness (/10) | 8.05 | **8.75** | 8.50 |
| Answer Relevancy (/10) | **8.90** | 8.60 | **8.90** |
| Context Recall (/10) | **9.00** | 8.50 | 8.30 |
| Tokens/sec | 4.43 | 5.97 | **7.90** |

---

### Key Findings

**1. Llama-3 leads on lexical and semantic similarity.**
Llama-3-8B achieved the highest scores on every lexical metric (ROUGE-1: 0.4564, ROUGE-L: 0.4072) and the highest BERTScore F1 (0.8351) and Semantic Similarity (0.6647). This reflects that its responses most closely match reference answers both in surface form and semantic meaning — a result of its larger 8B parameter backbone and strong instruction-following via the Llama chat template.

**2. Phi-3 leads on Faithfulness.**
Phi-3-Mini achieved the highest Faithfulness score (8.75/10), meaning its responses — when generated — were more consistently grounded in the retrieved context than either competing model. This is notable given its smaller size and suggests that its strict clinical system prompt is effective at preventing the model from introducing unsupported claims.

**3. Qwen3 leads on generation speed.**
At 7.90 tokens/sec, Qwen3 generates responses nearly twice as fast as Llama-3 (4.43 tokens/sec), making it the most practical choice for latency-sensitive deployment. Despite this speed advantage, it maintains fully competitive Answer Relevancy (8.90/10) and solid BERTScore F1 (0.8187).

**4. All three models show high RAG-judge scores, but with variance.**
The large standard deviations in Faithfulness, Answer Relevancy, and Context Recall (ranging from ±2.76 to ±3.64 across models) indicate that all three systems encounter edge-case queries where performance degrades — typically on highly specific procedural or table-dense questions within the ESC Guidelines. This points to a shared retrieval limitation rather than a model-specific generation weakness.

---

### Overall Recommendation

All three models are viable for a clinical RAG system grounded in the 2024 ESC Guidelines, each with a distinct strength profile:

- **Llama-3-8B-Instruct** — Best for accuracy-critical applications where lexical and semantic fidelity to reference answers is the priority.
- **Phi-3-Mini-4k-Instruct** — Best for hallucination-sensitive applications where grounding in retrieved context is non-negotiable, with the added benefit of fast retrieval.
- **Qwen3-4B-Instruct** — Best for real-time deployment where generation speed is essential without significantly sacrificing answer quality.

---

### Future Work
- Extend the evaluation set beyond 20 queries to improve statistical reliability, particularly for high-variance metrics like Context Recall.
- Explore domain-specific fine-tuning on ESC guideline QA pairs to raise the performance floor on edge-case queries across all three models.
- Investigate ensemble or routing strategies that direct queries to the most appropriate model based on query complexity or retrieval confidence.
- Benchmark all three models under identical GPU-optimized inference conditions (e.g., vLLM) for a controlled latency comparison.